# SciGeoGuard-X — Independent Validation v4

## Frozen verifier, blinded expert review, adjudication, confidence intervals, and optional open-source LLM baseline

This notebook is the **independent-validation stage** after the v3 engineering experiments.

It is intentionally different from v1–v3:

- the verifier is frozen before human evaluation;
- reviewer sheets hide generated labels, machine predictions, and truth explanations;
- two independent reviewers can label the same cases;
- disagreements are exported for adjudication;
- Cohen's kappa and raw agreement are computed;
- final SciGeoGuard-X metrics are computed only against adjudicated labels;
- confidence intervals, FSCR, proof coverage, and failure analysis are included;
- an optional zero-paid-API local LLM baseline is provided;
- reviewer and publication bundles are exported automatically.

**Important:** generated v3 stress tests and human-labelled independent validation are kept separate.

## 0. Recommended evaluation protocol

1. Freeze SciGeoGuard-X v3.
2. Produce blinded reviewer packs.
3. Reviewer A and Reviewer B independently assign `PROVEN`, `REFUTED`, or `UNPROVEN`.
4. Adjudicate disagreements.
5. Import the adjudicated labels.
6. Run the frozen verifier and baselines.
7. Report inter-rater agreement and post-freeze performance.
8. Keep generated-stress-test results separate from human-validation results.

In [ ]:
from pathlib import Path
from dataclasses import dataclass, field, asdict, replace
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional
import hashlib, json, math, random, shutil, zipfile

import numpy as np
import pandas as pd

SEED = 20261002
random.seed(SEED)
np.random.seed(SEED)

ROOT = Path("/content") if Path("/content").exists() else Path("/mnt/data")
WORK = ROOT / "scigeoguard_v4"
WORK.mkdir(parents=True, exist_ok=True)

print("SciGeoGuard-X Independent Validation v4")
print("ROOT =", ROOT)
print("WORK =", WORK)

# Part I — Freeze the v3 artifact

The notebook searches for the v3 notebook and result ZIP. If the v3 notebook is present, its SHA-256 digest becomes the primary freeze hash.

If only the result ZIP is present, the protocol still works, but the freeze record explicitly says `RESULTS_ONLY`.

In [ ]:
def find_first_exact_or_glob(exact_names, glob_patterns):
    roots = []
    for p in [ROOT, Path("/mnt/data"), Path("/content"), Path.cwd()]:
        if p.exists() and p not in roots:
            roots.append(p)

    for base in roots:
        for name in exact_names:
            p = base / name
            if p.exists():
                return p

    for base in roots:
        for pat in glob_patterns:
            hits = sorted(base.glob(pat))
            if hits:
                return hits[0]

    return None

v3_notebook = find_first_exact_or_glob(
    [
        "SciGeoGuard_X_Publication_Experiments_v3.ipynb",
        "SciGeoGuard_X_Publication_Experiments_v3_executed.ipynb",
    ],
    ["*SciGeoGuard*Publication*Experiments*v3*.ipynb"]
)

v3_zip = find_first_exact_or_glob(
    [
        "SciGeoGuard_X_v3_All_Results.zip",
        "SciGeoGuard_X_v3_All_Results (1).zip",
    ],
    ["*SciGeoGuard*v3*All*Results*.zip"]
)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

freeze_record = {
    "frozen_at_utc": datetime.now(timezone.utc).isoformat(),
    "verifier_version": "SciGeoGuard-X-v3",
    "v3_notebook": str(v3_notebook) if v3_notebook else None,
    "v3_results_zip": str(v3_zip) if v3_zip else None,
    "v3_notebook_sha256": sha256_file(v3_notebook) if v3_notebook else None,
    "v3_results_zip_sha256": sha256_file(v3_zip) if v3_zip else None,
    "freeze_status": (
        "VERIFIER_NOTEBOOK_HASHED" if v3_notebook
        else "RESULTS_ONLY" if v3_zip
        else "NO_V3_ARTIFACT_FOUND"
    )
}

freeze_path = WORK / "SciGeoGuard_X_v3_freeze_record.json"
freeze_path.write_text(json.dumps(freeze_record, indent=2), encoding="utf-8")

freeze_record

# Part II — Load the v3 benchmark

This benchmark is used only to create a **blinded human-review sample** and verify the evaluation machinery.

Reviewer files do not contain:

- generated `expected` verdict,
- generated `truth_reason`,
- SciGeoGuard-X prediction,
- verifier findings.

In [ ]:
V3_EXTRACT = WORK / "v3_results"
V3_EXTRACT.mkdir(exist_ok=True)

if v3_zip and v3_zip.exists():
    with zipfile.ZipFile(v3_zip, "r") as z:
        z.extractall(V3_EXTRACT)

candidate_bench = list(V3_EXTRACT.rglob("SilentGeoBench_v3.csv"))

if not candidate_bench:
    for base in [ROOT, Path("/mnt/data"), Path("/content")]:
        if base.exists():
            candidate_bench.extend(base.glob("**/SilentGeoBench_v3.csv"))

if candidate_bench:
    v3_bench_path = candidate_bench[0]
    v3 = pd.read_csv(v3_bench_path)
    print("Loaded v3 benchmark:", v3_bench_path)
    print("Shape:", v3.shape)
else:
    v3_bench_path = None
    v3 = pd.DataFrame()
    print("No v3 benchmark found.")

# Part III — Create a blinded expert-review sample

Default sample target: **180 cases**, stratified across v3 splits and families.

In [ ]:
BLIND_SAMPLE_N = 180

def stratified_blind_sample(df, n=180):
    if df.empty:
        return pd.DataFrame()

    required = {"case_id","split","tier","family","intent","artifacts","steps"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing benchmark columns: {sorted(missing)}")

    rng = np.random.default_rng(SEED)
    groups = list(df.groupby(["split","family"], dropna=False))

    selected = []

    # First take one from every group.
    for _, g in groups:
        selected.append(int(rng.choice(g.index)))

    # Fill remaining slots from the unused pool.
    selected = list(dict.fromkeys(selected))
    remaining = max(0, n-len(selected))
    pool = df.index.difference(pd.Index(selected))

    if remaining and len(pool):
        extra = rng.choice(pool, size=min(remaining,len(pool)), replace=False)
        selected.extend([int(x) for x in extra])

    sample = df.loc[selected].copy()
    sample = sample.sample(frac=1, random_state=SEED).reset_index(drop=True)
    sample["review_order"] = np.arange(1,len(sample)+1)
    return sample

blind_sample = stratified_blind_sample(v3, BLIND_SAMPLE_N)

print("Blinded sample size:", len(blind_sample))
if not blind_sample.empty:
    print("Distinct families:", blind_sample.family.nunique())
    print(blind_sample["split"].value_counts())

In [ ]:
REVIEW_COLUMNS = [
    "review_order",
    "case_id",
    "workflow_intent",
    "artifacts_json",
    "steps_json",
    "reviewer_verdict",
    "reviewer_error_class",
    "reviewer_severity",
    "reviewer_confidence_1_to_5",
    "reviewer_comments",
    "reviewer_id"
]

def make_review_pack(sample):
    if sample.empty:
        return pd.DataFrame(columns=REVIEW_COLUMNS)

    pack = pd.DataFrame({
        "review_order": sample["review_order"],
        "case_id": sample["case_id"],
        "workflow_intent": sample["intent"],
        "artifacts_json": sample["artifacts"],
        "steps_json": sample["steps"],
        "reviewer_verdict": "",
        "reviewer_error_class": "",
        "reviewer_severity": "",
        "reviewer_confidence_1_to_5": "",
        "reviewer_comments": "",
        "reviewer_id": ""
    })

    return pack[REVIEW_COLUMNS]

reviewer_a = make_review_pack(blind_sample)
reviewer_b = make_review_pack(blind_sample)

reviewer_a_path = WORK/"Reviewer_A_blind_cases.csv"
reviewer_b_path = WORK/"Reviewer_B_blind_cases.csv"

reviewer_a.to_csv(reviewer_a_path,index=False)
reviewer_b.to_csv(reviewer_b_path,index=False)

print(reviewer_a_path)
print(reviewer_b_path)

# Part IV — Template for genuinely new external workflows

This is more important than simply adding more generated mutations.

Ask reviewers/domain experts to author new workflows without seeing the SciGeoGuard-X rule implementation.

In [ ]:
EXTERNAL_CASES_TARGET = 120

external_template = pd.DataFrame({
    "external_case_id": [f"EXT_{i:03d}" for i in range(EXTERNAL_CASES_TARGET)],
    "domain": "",
    "scientific_goal": "",
    "artifact_1_semantics_json": "",
    "artifact_2_semantics_json": "",
    "workflow_steps_json": "",
    "scientific_assumptions_json": "",
    "source_or_reference": "",
    "case_author_id": "",
    "notes": ""
})

external_template_path = WORK/"External_Blind_Case_Authoring_Template.csv"
external_template.to_csv(external_template_path,index=False)

print(external_template_path)

# Part V — Reviewer import and inter-rater agreement

After reviewers finish, upload:

- `Reviewer_A_completed.csv`
- `Reviewer_B_completed.csv`

The notebook validates labels and computes raw agreement and Cohen's kappa.

In [ ]:
ALLOWED_VERDICTS = {"PROVEN","REFUTED","UNPROVEN"}

REVIEWER_A_COMPLETED = ROOT/"Reviewer_A_completed.csv"
REVIEWER_B_COMPLETED = ROOT/"Reviewer_B_completed.csv"

def load_review(path):
    if not path.exists():
        return None

    d = pd.read_csv(path,dtype=str).fillna("")
    required = {"case_id","reviewer_verdict"}

    if not required.issubset(d.columns):
        raise ValueError(f"{path.name} missing required columns.")

    d["reviewer_verdict"] = d["reviewer_verdict"].str.strip().str.upper()
    nonblank = d.reviewer_verdict.ne("")
    invalid = set(d.loc[nonblank,"reviewer_verdict"]) - ALLOWED_VERDICTS

    if invalid:
        raise ValueError(f"{path.name}: invalid verdicts {sorted(invalid)}")

    if d.case_id.duplicated().any():
        raise ValueError(f"{path.name}: duplicate case IDs.")

    return d

ra = load_review(REVIEWER_A_COMPLETED)
rb = load_review(REVIEWER_B_COMPLETED)

print("Reviewer A present:", ra is not None)
print("Reviewer B present:", rb is not None)

In [ ]:
def cohens_kappa(a,b,labels=("PROVEN","REFUTED","UNPROVEN")):
    a = pd.Series(a)
    b = pd.Series(b)

    if len(a)==0:
        return np.nan

    po = float((a.values==b.values).mean())
    pa = a.value_counts(normalize=True)
    pb = b.value_counts(normalize=True)
    pe = sum(float(pa.get(x,0))*float(pb.get(x,0)) for x in labels)

    if math.isclose(1-pe,0):
        return np.nan

    return (po-pe)/(1-pe)

agreement_summary = None
reviewer_disagreements = pd.DataFrame()

if ra is not None and rb is not None:
    common = ra[["case_id","reviewer_verdict","reviewer_comments"]].merge(
        rb[["case_id","reviewer_verdict","reviewer_comments"]],
        on="case_id",
        suffixes=("_a","_b")
    )

    common = common[
        common.reviewer_verdict_a.isin(ALLOWED_VERDICTS) &
        common.reviewer_verdict_b.isin(ALLOWED_VERDICTS)
    ].copy()

    agreement_summary = {
        "n_double_reviewed": int(len(common)),
        "raw_agreement": float(
            (common.reviewer_verdict_a==common.reviewer_verdict_b).mean()
        ) if len(common) else np.nan,
        "cohens_kappa": float(
            cohens_kappa(common.reviewer_verdict_a,common.reviewer_verdict_b)
        ) if len(common) else np.nan
    }

    reviewer_disagreements = common[
        common.reviewer_verdict_a != common.reviewer_verdict_b
    ].copy()

    print(agreement_summary)
    print("Disagreements:",len(reviewer_disagreements))
else:
    print("Agreement analysis is ready; completed reviewer files are not present yet.")

# Part VI — Adjudication

Agreed reviewer labels can be retained. Disagreements are exported for a third reviewer or consensus adjudication.

In [ ]:
adjudication_path = WORK/"Adjudication_required.csv"
agreements_path = WORK/"Reviewer_agreements_provisional.csv"

if ra is not None and rb is not None:
    common = ra[["case_id","reviewer_verdict","reviewer_comments"]].merge(
        rb[["case_id","reviewer_verdict","reviewer_comments"]],
        on="case_id",
        suffixes=("_a","_b")
    )

    agreed = common[
        common.reviewer_verdict_a == common.reviewer_verdict_b
    ].copy()
    agreed["adjudicated_verdict"] = agreed["reviewer_verdict_a"]

    disagree = common[
        common.reviewer_verdict_a != common.reviewer_verdict_b
    ].copy()
    disagree["adjudicated_verdict"] = ""
    disagree["adjudicator_id"] = ""
    disagree["adjudication_reason"] = ""

    agreed.to_csv(agreements_path,index=False)
    disagree.to_csv(adjudication_path,index=False)

    print("Agreements:", agreements_path)
    print("Adjudication required:", adjudication_path)
else:
    pd.DataFrame(columns=[
        "case_id","reviewer_verdict_a","reviewer_verdict_b",
        "adjudicated_verdict","adjudicator_id","adjudication_reason"
    ]).to_csv(adjudication_path,index=False)

    print("Empty adjudication template:",adjudication_path)

# Part VII — Frozen SciGeoGuard-X v3 verifier

This is embedded so the independent-validation notebook remains standalone.

**Do not edit this block after reviewer labels are collected.**

In [ ]:
@dataclass
class Artifact:
    name: str
    quantity: str
    unit: Optional[str] = None
    scale: str = "continuous"
    crs: Optional[str] = None
    resolution_m: Optional[float] = None
    temporal_resolution_h: Optional[float] = None
    processing_level: Optional[str] = None
    surface_model: Optional[str] = None
    vertical_datum: Optional[str] = None
    acquisition_time: Optional[str] = None
    available_time: Optional[str] = None
    provenance: Optional[str] = None
    band_role: Optional[str] = None
    nodata_semantics: Optional[str] = None
    uncertainty: Optional[float] = None
    allowed_values: Optional[List[float]] = None
    observed_min: Optional[float] = None
    observed_max: Optional[float] = None
    attrs: Dict[str,Any] = field(default_factory=dict)

@dataclass
class Step:
    op: str
    params: Dict[str,Any] = field(default_factory=dict)

@dataclass
class Intent:
    goal: str
    required_quantity: Optional[str] = None
    required_unit: Optional[str] = None
    decision_time: Optional[str] = None
    max_resolution_m: Optional[float] = None
    requires_bare_earth: bool = False
    requires_cloud_mask: bool = False
    max_uncertainty: Optional[float] = None
    temporal_tolerance_h: Optional[float] = None
    required_value_range: Optional[List[float]] = None

RATE_UNITS={"mm/h","mm/hr","MM/Hr"}
CATEGORICAL_SCALES={"categorical","nominal","ordinal","binary"}
CONTINUOUS_INTERPOLATORS={"bilinear","cubic","lanczos"}

UNIT_DIMENSION={
    "m2":"area","km2":"area","ha":"area",
    "m":"length","km":"length","cm":"length","mm":"length_or_accum",
    "mm/h":"rate","mm/hr":"rate","MM/Hr":"rate",
    "dimensionless":"dimensionless","1":"dimensionless",
    "K":"temperature","degC":"temperature"
}

def parse_dt(x):
    if not x:
        return None
    d=datetime.fromisoformat(str(x).replace("Z","+00:00"))
    if d.tzinfo is None:
        d=d.replace(tzinfo=timezone.utc)
    return d

def unit_dimension(u):
    return UNIT_DIMENSION.get(u)

def frozen_verify(artifacts,steps,intent):
    A=[replace(x,attrs=dict(x.attrs)) for x in artifacts]
    findings=[]

    def add(code,verdict,message,step_index=None):
        findings.append({
            "code":code,
            "verdict":verdict,
            "message":message,
            "step_index":step_index
        })

    for x in A:
        if not x.provenance:
            add("PROVENANCE_MISSING","UNPROVEN","Source provenance is missing.")
        if x.resolution_m is not None and x.resolution_m<=0:
            add("RESOLUTION_INVALID","REFUTED","Spatial resolution must be positive.")

    for si,s in enumerate(steps):
        p=s.params

        if s.op=="resample":
            x=A[p["artifact"]]
            method=str(p.get("method","nearest")).lower()
            target=float(p["target_resolution_m"])

            if x.scale in CATEGORICAL_SCALES and method in CONTINUOUS_INTERPOLATORS:
                add("CATEGORICAL_INTERPOLATION","REFUTED",
                    "Continuous interpolation does not preserve categorical identity.",si)

            if x.resolution_m is not None and target<x.resolution_m:
                gain=x.resolution_m/target
                if gain>1.25 and not p.get("evidence_of_new_information",False):
                    add("UNSUPPORTED_RESOLUTION_GAIN","REFUTED",
                        "Finer grid spacing is created without evidence of new information.",si)

            x.resolution_m=target

        elif s.op=="temporal_aggregate":
            x=A[p["artifact"]]
            method=str(p.get("method","")).lower()
            duration=p.get("sample_duration_h")

            if x.unit in RATE_UNITS and p.get("declared_quantity")=="accumulation":
                if method in {"mean","average"}:
                    add("RATE_AVERAGED_AS_ACCUMULATION","REFUTED",
                        "A rate is averaged but declared as an accumulation.",si)
                elif method=="sum" and duration not in {1,1.0}:
                    add("RATE_SUM_WITHOUT_DURATION","REFUTED",
                        "Sub-hourly rates are summed without duration weighting.",si)

        elif s.op=="area":
            x=A[p["artifact"]]
            if x.crs in {"EPSG:4326","OGC:CRS84","CRS84"} and p.get("method","planar")=="planar":
                add("ANGULAR_COORDINATE_AREA","REFUTED",
                    "Planar metric area is computed directly from angular coordinates.",si)

        elif s.op=="spectral_index":
            red=A[p["red_artifact"]]
            other=A[p["other_artifact"]]

            if red.band_role!="red":
                add("RED_ROLE_MISMATCH","REFUTED","Red spectral role required.",si)

            expected=p.get("expected_other_role")
            if expected and other.band_role!=expected:
                add("SPECTRAL_ROLE_MISMATCH","REFUTED",
                    "Spectral operand role is incompatible with the requested index.",si)

            if (red.resolution_m is not None and other.resolution_m is not None
                and red.resolution_m!=other.resolution_m
                and not p.get("explicit_alignment",False)):
                add("BAND_GRID_MISMATCH","REFUTED",
                    "Spectral operands have different grids without explicit alignment.",si)

        elif s.op=="require_surface_reflectance":
            x=A[p["artifact"]]
            if x.processing_level not in {"L2A","surface_reflectance"}:
                add("PROCESSING_LEVEL_MISMATCH","REFUTED",
                    "Surface reflectance is required.",si)

        elif s.op=="require_cloud_mask":
            x=A[p["artifact"]]
            if not bool(x.attrs.get("cloud_mask_applied",False)):
                add("CLOUD_MASK_MISSING","REFUTED",
                    "Cloud-sensitive analysis lacks a cloud mask.",si)

        elif s.op=="require_bare_earth":
            x=A[p["artifact"]]
            if x.surface_model=="DSM":
                add("DSM_USED_AS_DTM","REFUTED",
                    "Bare-earth analysis is requested from a DSM.",si)
            elif x.surface_model is None:
                add("SURFACE_MODEL_UNKNOWN","UNPROVEN",
                    "DSM/DTM semantics are unknown.",si)

        elif s.op=="combine_vertical":
            x,y=A[p["a"]],A[p["b"]]

            if not x.vertical_datum or not y.vertical_datum:
                add("VERTICAL_DATUM_UNKNOWN","UNPROVEN",
                    "Vertical-datum compatibility cannot be established.",si)
            elif x.vertical_datum!=y.vertical_datum:
                add("VERTICAL_DATUM_MISMATCH","REFUTED",
                    "Vertical quantities use incompatible datums.",si)

        elif s.op=="predict":
            decision=parse_dt(intent.decision_time)

            for i in p.get("features",range(len(A))):
                t=parse_dt(A[i].available_time or A[i].acquisition_time)

                if decision is not None and t is None:
                    add("FEATURE_TIME_UNKNOWN","UNPROVEN",
                        "Feature availability time is missing.",si)
                elif decision is not None and t>decision:
                    add("FUTURE_INFORMATION","REFUTED",
                        "A feature becomes available after the decision time.",si)

        elif s.op=="split":
            mode=p.get("mode")

            if mode=="spatial":
                if float(p.get("separation_m",0)) < float(p.get("dependence_range_m",0)):
                    add("SPATIAL_DEPENDENCE_LEAKAGE","REFUTED",
                        "Train/test separation is below the dependence range.",si)

            elif mode=="temporal":
                train_end=parse_dt(p.get("train_end"))
                test_start=parse_dt(p.get("test_start"))

                if train_end and test_start and train_end>=test_start:
                    add("TEMPORAL_SPLIT_OVERLAP","REFUTED",
                        "Training and test periods overlap.",si)

            elif mode=="scene":
                if float(p.get("scene_overlap_fraction",0))>0:
                    add("SCENE_OVERLAP_LEAKAGE","REFUTED",
                        "Scene content appears in both train and test.",si)

        elif s.op=="fill_nodata":
            x=A[p["artifact"]]
            if (
                x.nodata_semantics in {"missing","masked"}
                and p.get("value")==0
                and p.get("zero_is_physically_valid",True)
            ):
                add("NODATA_COLLAPSED_TO_ZERO","REFUTED",
                    "Missing data are replaced by a valid physical zero.",si)

        elif s.op=="compare_sensors":
            x,y=A[p["a"]],A[p["b"]]
            tx,ty=parse_dt(x.acquisition_time),parse_dt(y.acquisition_time)
            tol=float(p.get("tolerance_h",intent.temporal_tolerance_h or 0))

            if not tx or not ty:
                add("ACQUISITION_TIME_UNKNOWN","UNPROVEN",
                    "Acquisition-time compatibility is unknown.",si)
            else:
                gap=abs((tx-ty).total_seconds())/3600
                if gap>tol:
                    add("MULTISENSOR_TIME_MISMATCH","REFUTED",
                        "Acquisition gap exceeds the declared tolerance.",si)

        elif s.op=="convert_unit":
            x=A[p["artifact"]]
            ds,dd=unit_dimension(x.unit),unit_dimension(p["to"])

            if ds is None or dd is None:
                add("UNIT_DIMENSION_UNKNOWN","UNPROVEN",
                    "Unit dimensionality is unknown.",si)
            elif ds!=dd and not ({ds,dd}<={"length","length_or_accum"}):
                add("UNIT_DIMENSION_MISMATCH","REFUTED",
                    "Requested conversion changes physical dimension.",si)

        elif s.op=="assert_value_range":
            x=A[p["artifact"]]
            lo,hi=p["min"],p["max"]

            if x.observed_min is None or x.observed_max is None:
                add("VALUE_RANGE_UNOBSERVED","UNPROVEN",
                    "Physical value bounds cannot be checked.",si)
            elif x.observed_min<lo or x.observed_max>hi:
                add("PHYSICAL_RANGE_VIOLATION","REFUTED",
                    "Observed values violate physical bounds.",si)

        elif s.op=="aggregate_categorical":
            x=A[p["artifact"]]
            if x.scale in CATEGORICAL_SCALES and p.get("method") in {"mean","median"}:
                add("CATEGORICAL_AGGREGATION_INVALID","REFUTED",
                    "Nominal labels cannot be averaged meaningfully.",si)

        elif s.op=="drop_quality_metadata":
            if p.get("required_downstream",False):
                add("QUALITY_METADATA_LOST","REFUTED",
                    "Required quality metadata were discarded.",si)

    if intent.max_uncertainty is not None:
        known=[x.uncertainty for x in A if x.uncertainty is not None]

        if not known:
            add("UNCERTAINTY_UNKNOWN","UNPROVEN",
                "Required uncertainty evidence is missing.")
        elif max(known)>intent.max_uncertainty:
            add("UNCERTAINTY_EXCEEDS_LIMIT","REFUTED",
                "Uncertainty exceeds the declared limit.")

    if any(f["verdict"]=="REFUTED" for f in findings):
        verdict="REFUTED"
    elif any(f["verdict"]=="UNPROVEN" for f in findings):
        verdict="UNPROVEN"
    else:
        verdict="PROVEN"

    return verdict,findings

# Part VIII — Parse v3 workflow JSON and run the frozen verifier

In [ ]:
def parse_artifacts_json(s):
    return [Artifact(**x) for x in json.loads(s)]

def parse_steps_json(s):
    return [Step(**x) for x in json.loads(s)]

def parse_intent_json(s):
    return Intent(**json.loads(s))

def frozen_predict_v3_row(row):
    artifacts=parse_artifacts_json(row["artifacts"])
    steps=parse_steps_json(row["steps"])
    intent=parse_intent_json(row["intent"])
    return frozen_verify(artifacts,steps,intent)

pilot_results=pd.DataFrame()

if not blind_sample.empty:
    rows=[]
    for _,r in blind_sample.iterrows():
        pred,findings=frozen_predict_v3_row(r)

        rows.append({
            "case_id":r["case_id"],
            "frozen_prediction":pred,
            "generated_expected":r.get("expected",""),
            "matches_generated_label":pred==r.get("expected",""),
            "n_findings":len(findings)
        })

    pilot_results=pd.DataFrame(rows)

    print("Pilot sample:",len(pilot_results))
    print(
        "Agreement with generated labels:",
        round(float(pilot_results.matches_generated_label.mean()),4)
    )
else:
    print("Pilot skipped: v3 benchmark unavailable.")

# Part IX — Optional free/open-source LLM baseline

This is disabled by default.

It uses a local Hugging Face model in Colab rather than a paid API. You can change the model ID if desired.

In [ ]:
RUN_LLM_BASELINE = False
LLM_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
LLM_MAX_CASES = 180

def make_llm_prompt(row):
    return (
        "Judge the scientific validity of this Earth-observation workflow.\n\n"
        "Return ONLY JSON with keys verdict and reason.\n"
        "verdict must be exactly PROVEN, REFUTED, or UNPROVEN.\n\n"
        "PROVEN = supplied evidence supports the scientific goal.\n"
        "REFUTED = at least one scientific constraint is violated.\n"
        "UNPROVEN = validity cannot be established because needed evidence is missing.\n\n"
        f"Scientific goal:\n{row['workflow_intent']}\n\n"
        f"Artifacts:\n{row['artifacts_json']}\n\n"
        f"Workflow steps:\n{row['steps_json']}\n"
    )

print("LLM baseline enabled:",RUN_LLM_BASELINE)
print("Model:",LLM_MODEL_ID)

In [ ]:
def run_local_llm_baseline(pack):
    if not RUN_LLM_BASELINE:
        print("LLM baseline disabled.")
        return pd.DataFrame()

    import sys,subprocess
    subprocess.check_call([
        sys.executable,"-m","pip","install","-q",
        "transformers","accelerate","sentencepiece"
    ])

    import torch
    from transformers import AutoTokenizer,AutoModelForCausalLM

    tok=AutoTokenizer.from_pretrained(LLM_MODEL_ID)
    model=AutoModelForCausalLM.from_pretrained(
        LLM_MODEL_ID,
        torch_dtype="auto",
        device_map="auto"
    )

    rows=[]

    for _,r in pack.head(LLM_MAX_CASES).iterrows():
        prompt=make_llm_prompt(r)
        messages=[{"role":"user","content":prompt}]

        rendered=tok.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inp=tok([rendered],return_tensors="pt").to(model.device)

        generated=model.generate(
            **inp,
            max_new_tokens=120,
            do_sample=False
        )

        answer=tok.decode(
            generated[0][inp.input_ids.shape[1]:],
            skip_special_tokens=True
        )

        upper=answer.upper()
        verdict="UNPARSED"

        for label in ["PROVEN","REFUTED","UNPROVEN"]:
            if label in upper:
                verdict=label
                break

        rows.append({
            "case_id":r["case_id"],
            "llm_model":LLM_MODEL_ID,
            "llm_verdict":verdict,
            "raw_output":answer
        })

    return pd.DataFrame(rows)

# To run later:
# llm_results = run_local_llm_baseline(reviewer_a)

# Part X — Import adjudicated labels

After adjudication, upload a CSV named:

`Adjudicated_labels.csv`

with:

- `case_id`
- `adjudicated_verdict`

In [ ]:
ADJUDICATED_FILE = ROOT/"Adjudicated_labels.csv"

def load_adjudicated(path):
    if not path.exists():
        return None

    d=pd.read_csv(path,dtype=str).fillna("")

    required={"case_id","adjudicated_verdict"}
    if not required.issubset(d.columns):
        raise ValueError("Adjudicated_labels.csv is missing required columns.")

    d["adjudicated_verdict"]=d["adjudicated_verdict"].str.strip().str.upper()

    invalid=set(d.adjudicated_verdict)-ALLOWED_VERDICTS
    if invalid:
        raise ValueError(f"Invalid adjudicated labels: {sorted(invalid)}")

    if d.case_id.duplicated().any():
        raise ValueError("Duplicate adjudicated case IDs.")

    return d

adjudicated=load_adjudicated(ADJUDICATED_FILE)
print("Adjudicated labels present:",adjudicated is not None)

# Part XI — Post-freeze metrics

Metrics include:

- exact 3-state accuracy;
- FSCR;
- false-refutation rate;
- proof coverage;
- valid-certification rate;
- bootstrap 95% confidence intervals.

In [ ]:
def metric_point(df,pred_col,truth_col="truth"):
    truth=df[truth_col]
    pred=df[pred_col]

    invalid=truth.eq("REFUTED")
    valid=truth.eq("PROVEN")

    return {
        "accuracy":float((truth==pred).mean()),
        "FSCR":float((pred[invalid]=="PROVEN").mean()) if invalid.any() else np.nan,
        "false_refutation_rate":float((pred[valid]=="REFUTED").mean()) if valid.any() else np.nan,
        "proof_coverage":float(pred.isin(["PROVEN","REFUTED"]).mean()),
        "valid_certification_rate":float((pred[valid]=="PROVEN").mean()) if valid.any() else np.nan
    }

def bootstrap_ci(df,pred_col,metric_name,B=1000):
    if len(df)==0:
        return np.nan,np.nan,np.nan

    rng=np.random.default_rng(SEED)
    vals=[]

    for _ in range(B):
        samp=df.iloc[rng.integers(0,len(df),len(df))]
        value=metric_point(samp,pred_col).get(metric_name,np.nan)

        if value==value:
            vals.append(value)

    point=metric_point(df,pred_col).get(metric_name,np.nan)

    if not vals:
        return point,np.nan,np.nan

    lo,hi=np.percentile(vals,[2.5,97.5])
    return float(point),float(lo),float(hi)

print("Metric functions ready.")

In [ ]:
final_eval=pd.DataFrame()
final_metrics=pd.DataFrame()

if adjudicated is not None and not blind_sample.empty:
    evaluation=blind_sample.merge(
        adjudicated,
        on="case_id",
        how="inner"
    )

    rows=[]

    for _,r in evaluation.iterrows():
        pred,findings=frozen_predict_v3_row(r)
        artifacts=parse_artifacts_json(r["artifacts"])

        runtime="PROVEN"
        metadata_only=(
            "UNPROVEN"
            if any(x.provenance is None for x in artifacts)
            else "PROVEN"
        )

        rows.append({
            "case_id":r["case_id"],
            "truth":r["adjudicated_verdict"],
            "runtime":runtime,
            "metadata_only":metadata_only,
            "frozen_scigeoguard":pred,
            "n_findings":len(findings)
        })

    final_eval=pd.DataFrame(rows)

    metric_rows=[]

    for system in ["runtime","metadata_only","frozen_scigeoguard"]:
        row={"system":system,**metric_point(final_eval,system)}

        for m in ["accuracy","FSCR","proof_coverage"]:
            _,lo,hi=bootstrap_ci(final_eval,system,m)
            row[f"{m}_ci_low"]=lo
            row[f"{m}_ci_high"]=hi

        metric_rows.append(row)

    final_metrics=pd.DataFrame(metric_rows)
    final_metrics
else:
    print("Final human-labelled evaluation awaits Adjudicated_labels.csv.")

# Part XII — Failure analysis

In [ ]:
false_certifications=pd.DataFrame()
false_refutations=pd.DataFrame()
unproven_outputs=pd.DataFrame()

if not final_eval.empty:
    false_certifications=final_eval[
        (final_eval.truth=="REFUTED") &
        (final_eval.frozen_scigeoguard=="PROVEN")
    ].copy()

    false_refutations=final_eval[
        (final_eval.truth=="PROVEN") &
        (final_eval.frozen_scigeoguard=="REFUTED")
    ].copy()

    unproven_outputs=final_eval[
        final_eval.frozen_scigeoguard=="UNPROVEN"
    ].copy()

    print("False scientific certifications:",len(false_certifications))
    print("False refutations:",len(false_refutations))
    print("UNPROVEN outputs:",len(unproven_outputs))
else:
    print("Failure analysis awaits adjudicated labels.")

# Part XIII — Protocol leakage checks

In [ ]:
checks = {
    "freeze_record_exists": freeze_path.exists(),
    "reviewer_A_hides_expected": "expected" not in reviewer_a.columns,
    "reviewer_B_hides_expected": "expected" not in reviewer_b.columns,
    "reviewer_A_hides_machine_prediction": not any(
        "prediction" in c.lower() or "verifier" in c.lower()
        for c in reviewer_a.columns
    ),
    "reviewer_B_hides_machine_prediction": not any(
        "prediction" in c.lower() or "verifier" in c.lower()
        for c in reviewer_b.columns
    ),
    "reviewer_A_unique_case_ids": not reviewer_a.case_id.duplicated().any() if len(reviewer_a) else True,
    "reviewer_B_unique_case_ids": not reviewer_b.case_id.duplicated().any() if len(reviewer_b) else True,
    "external_template_has_no_machine_label": not any(
        "expected" in c.lower() or "machine" in c.lower()
        for c in external_template.columns
    )
}

protocol_df=pd.DataFrame([
    {"check":k,"passed":bool(v)}
    for k,v in checks.items()
])

protocol_df

In [ ]:
assert protocol_df.passed.all(), "Protocol leakage/quality gate failed."
print("All protocol-quality gates passed.")

# Part XIV — Export publication and reviewer packages

In [ ]:
protocol_df.to_csv(WORK/"Protocol_quality_checks.csv",index=False)
pilot_results.to_csv(WORK/"Pilot_frozen_verifier_check.csv",index=False)
external_template.to_csv(WORK/"External_Blind_Case_Authoring_Template.csv",index=False)

if agreement_summary is not None:
    pd.DataFrame([agreement_summary]).to_csv(
        WORK/"Inter_rater_agreement.csv",
        index=False
    )
    reviewer_disagreements.to_csv(
        WORK/"Reviewer_disagreements.csv",
        index=False
    )

if not final_eval.empty:
    final_eval.to_csv(
        WORK/"Independent_validation_case_results.csv",
        index=False
    )
    final_metrics.to_csv(
        WORK/"Independent_validation_metrics.csv",
        index=False
    )
    false_certifications.to_csv(
        WORK/"False_scientific_certifications.csv",
        index=False
    )
    false_refutations.to_csv(
        WORK/"False_refutations.csv",
        index=False
    )
    unproven_outputs.to_csv(
        WORK/"Unproven_outputs.csv",
        index=False
    )

readme_text = "\n".join([
    "SciGeoGuard-X Independent Validation v4",
    "="*45,
    "",
    "Freeze status:",
    json.dumps(freeze_record,indent=2),
    "",
    "Protocol:",
    "1. Two reviewers independently label blinded cases.",
    "2. Allowed labels: PROVEN / REFUTED / UNPROVEN.",
    "3. Reviewer sheets contain no machine prediction or generated expected label.",
    "4. Disagreements require adjudication.",
    "5. Frozen-verifier scoring is performed only against adjudicated labels.",
    "6. The v3 pilot reproduction is not independent human validation.",
    "",
    f"Reviewer A loaded: {ra is not None}",
    f"Reviewer B loaded: {rb is not None}",
    f"Adjudicated labels loaded: {adjudicated is not None}",
])

readme = WORK/"README_VALIDATION_PROTOCOL.txt"
readme.write_text(readme_text,encoding="utf-8")

print("Exports written to",WORK)

In [ ]:
reviewer_dir=WORK/"reviewer_package"

if reviewer_dir.exists():
    shutil.rmtree(reviewer_dir)

reviewer_dir.mkdir()

for p in [
    reviewer_a_path,
    reviewer_b_path,
    external_template_path,
    readme
]:
    if p.exists():
        shutil.copy2(p,reviewer_dir/p.name)

reviewer_zip = shutil.make_archive(
    str(WORK/"SciGeoGuard_X_v4_Reviewer_Package"),
    "zip",
    root_dir=reviewer_dir
)

full_zip = shutil.make_archive(
    str(ROOT/"SciGeoGuard_X_v4_Independent_Validation_Bundle"),
    "zip",
    root_dir=WORK
)

print("Reviewer package:",reviewer_zip)
print("Full validation bundle:",full_zip)

# What comes after this notebook

Once reviewers return labels:

1. put `Reviewer_A_completed.csv` and `Reviewer_B_completed.csv` in Colab;
2. run the agreement/adjudication cells;
3. create `Adjudicated_labels.csv`;
4. rerun the post-freeze metric cells;
5. optionally enable the open-source LLM baseline;
6. report human-labelled independent results separately from v3 generated stress tests.

That is the experiment needed before making strong generalization claims in the manuscript.